# byGPT

In [1]:
############################################
# GPT-1 스타일 Language Model 구현
# 논문: Improving Language Understanding by Generative Pre-Training
#
# 기존 Transformer Encoder-Decoder 구조와 달리
# GPT는 Decoder-only Transformer 구조를 사용한다.
#
# 주요 변경점
# 1. Encoder 제거
# 2. Masked Self-Attention 사용 (미래 토큰 차단)
# 3. Language Modeling 방식 학습
############################################

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import re
from tqdm import tqdm

In [3]:
############################################
# GPU 사용 여부 확인
############################################

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else ('mps' if torch.mps.is_available() else 'cpu'))

In [5]:
############################################
# 1. 데이터 전처리
#
# GPT는 번역 모델이 아니라
# "Language Modeling" 방식으로 학습한다.
#
# 즉 하나의 문장을 입력받아
# 다음 단어를 예측하도록 학습한다.
#
# 예:
# 입력  : I love deep learning
# 출력  : love deep learning <eos>
############################################

In [6]:
def preprocess_sentence(sentence):

    # 소문자로 변환
    sentence = sentence.lower()

    # 알파벳과 일부 문장부호만 남김
    sentence = re.sub(r'[^a-zA-Z?.!,]+', ' ', sentence)

    # 공백 정리
    sentence = re.sub(r' {2,}', ' ', sentence)

    return sentence.strip()

In [7]:
############################################
# spa-eng 데이터셋 사용
# (영어 문장만 사용하여 language modeling 수행)
############################################

In [8]:
file_path = "/Users/chaliepark/Documents/모두의연구소_아이펠_ai엔지니어/교육내용/s2s_translation/datasets/spa-eng/spa-eng/spa.txt"

with open(file_path, "r", encoding="utf-8") as f:
    lines = f.read().splitlines()

sentences = []

for line in lines:

    # spa.txt 구조
    # 영어문장 \t 스페인어문장
    eng = line.split('\t')[0]

    sentences.append(preprocess_sentence(eng))

print("dataset size:", len(sentences))

dataset size: 118964


In [9]:
############################################
# 2. Vocabulary 생성
#
# 단어 단위 토큰화 수행
############################################

In [10]:
words = set()

for s in sentences:

    for w in s.split():

        words.add(w)

# 특수 토큰 추가
vocab = ["<pad>", "<bos>", "<eos>"] + list(words)

stoi = {w:i for i,w in enumerate(vocab)}
itos = {i:w for w,i in stoi.items()}

vocab_size = len(vocab)

print("vocab size:", vocab_size)

vocab size: 22420


In [11]:
############################################
# 3. 문장을 토큰 인덱스로 변환
#
# 예:
# <bos> i love ai <eos>
############################################

In [12]:
max_len = 40

def encode(sentence):

    tokens = [stoi["<bos>"]]

    for w in sentence.split():
        tokens.append(stoi[w])

    tokens.append(stoi["<eos>"])

    # padding
    if len(tokens) < max_len:

        tokens += [stoi["<pad>"]] * (max_len-len(tokens))

    else:

        tokens = tokens[:max_len]

    return tokens


data = [encode(s) for s in sentences]

In [13]:
############################################
# 4. GPT Dataset 구성
#
# GPT 학습 방식
#
# input  : token[0 ... n-1]
# target : token[1 ... n]
#
# 즉 다음 단어 예측 문제
############################################

In [14]:
class GPTDataset(torch.utils.data.Dataset):

    def __init__(self,data):

        self.data=data

    def __len__(self):

        return len(self.data)

    def __getitem__(self,idx):

        tokens=self.data[idx]

        # 입력
        x=torch.tensor(tokens[:-1])

        # 정답
        y=torch.tensor(tokens[1:])

        return x,y


dataset = GPTDataset(data)

loader = torch.utils.data.DataLoader(
    dataset,
    batch_size=32,
    shuffle=True
)

In [15]:
############################################
# 5. Positional Embedding
#
# Transformer는 순서 정보가 없기 때문에
# 위치 정보를 embedding으로 추가해야 한다.
############################################

In [16]:
class PositionalEmbedding(nn.Module):

    def __init__(self,max_len,d_model):

        super().__init__()

        # 위치별 embedding
        self.embedding = nn.Embedding(max_len,d_model)

    def forward(self,x):

        B,T = x.shape

        # position index 생성
        pos = torch.arange(T,device=x.device)

        pos = self.embedding(pos)

        return pos

In [17]:
############################################
# 6. Causal Self Attention
#
# GPT의 핵심
#
# 미래 토큰을 보지 못하도록
# triangular mask 적용
#
# 예
#
# token1 -> token1
# token2 -> token1, token2
# token3 -> token1, token2, token3
############################################

In [18]:
class CausalSelfAttention(nn.Module):

    def __init__(self,d_model,heads):

        super().__init__()

        self.heads=heads
        self.head_dim=d_model//heads

        # QKV projection
        self.q=nn.Linear(d_model,d_model)
        self.k=nn.Linear(d_model,d_model)
        self.v=nn.Linear(d_model,d_model)

        # output projection
        self.proj=nn.Linear(d_model,d_model)

    def forward(self,x):

        B,T,C=x.shape

        q=self.q(x)
        k=self.k(x)
        v=self.v(x)

        # multi-head reshape
        q=q.view(B,T,self.heads,self.head_dim).transpose(1,2)
        k=k.view(B,T,self.heads,self.head_dim).transpose(1,2)
        v=v.view(B,T,self.heads,self.head_dim).transpose(1,2)

        # attention score
        att=(q @ k.transpose(-2,-1))/math.sqrt(self.head_dim)

        ############################################
        # Causal Mask
        #
        # 미래 단어 attention 차단
        ############################################

        mask=torch.tril(torch.ones(T,T,device=x.device))

        att=att.masked_fill(mask==0,-1e9)

        att=F.softmax(att,dim=-1)

        out=att @ v

        out=out.transpose(1,2).contiguous().view(B,T,C)

        return self.proj(out)


In [19]:
############################################
# 7. GPT Block
#
# 구조
#
# LayerNorm
# SelfAttention
# Residual
#
# LayerNorm
# FeedForward
# Residual
############################################

In [20]:
class GPTBlock(nn.Module):

    def __init__(self,d_model,heads):

        super().__init__()

        self.ln1=nn.LayerNorm(d_model)

        self.attn=CausalSelfAttention(d_model,heads)

        self.ln2=nn.LayerNorm(d_model)

        self.mlp=nn.Sequential(
            nn.Linear(d_model,4*d_model),
            nn.GELU(),
            nn.Linear(4*d_model,d_model)
        )

    def forward(self,x):

        x=x+self.attn(self.ln1(x))

        x=x+self.mlp(self.ln2(x))

        return x

In [21]:
############################################
# 8. GPT Model
#
# 구조
#
# Token Embedding
# Position Embedding
# GPT Blocks
# LayerNorm
# LM Head
############################################

In [22]:
class GPT(nn.Module):

    def __init__(self,
                 vocab_size,
                 max_len=40,
                 layers=6,
                 d_model=256,
                 heads=8):

        super().__init__()

        # token embedding
        self.token_embed=nn.Embedding(vocab_size,d_model)

        # position embedding
        self.pos_embed=PositionalEmbedding(max_len,d_model)

        # transformer blocks
        self.blocks=nn.Sequential(
            *[GPTBlock(d_model,heads) for _ in range(layers)]
        )

        self.ln=nn.LayerNorm(d_model)

        # language modeling head
        self.head=nn.Linear(d_model,vocab_size)

    def forward(self,x):

        tok=self.token_embed(x)

        pos=self.pos_embed(x)

        x=tok+pos

        x=self.blocks(x)

        x=self.ln(x)

        logits=self.head(x)

        return logits

In [23]:
############################################
# 9. 모델 생성
############################################

In [24]:
model = GPT(vocab_size).to(device)

print(model)


GPT(
  (token_embed): Embedding(22420, 256)
  (pos_embed): PositionalEmbedding(
    (embedding): Embedding(40, 256)
  )
  (blocks): Sequential(
    (0): GPTBlock(
      (ln1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (attn): CausalSelfAttention(
        (q): Linear(in_features=256, out_features=256, bias=True)
        (k): Linear(in_features=256, out_features=256, bias=True)
        (v): Linear(in_features=256, out_features=256, bias=True)
        (proj): Linear(in_features=256, out_features=256, bias=True)
      )
      (ln2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (mlp): Sequential(
        (0): Linear(in_features=256, out_features=1024, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=1024, out_features=256, bias=True)
      )
    )
    (1): GPTBlock(
      (ln1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (attn): CausalSelfAttention(
        (q): Linear(in_features=256, out_features=256, bias=T

In [25]:
############################################
# 10. Training
#
# Loss
# CrossEntropyLoss
############################################

In [ ]:
optimizer = torch.optim.Adam(model.parameters(),lr=3e-4)

criterion = nn.CrossEntropyLoss()

for epoch in range(3):

    total_loss=0

    for x,y in tqdm(loader):

        x=x.to(device)
        y=y.to(device)

        logits=model(x)

        B,T,C=logits.shape

        loss=criterion(
            logits.view(B*T,C),
            y.view(B*T)
        )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss+=loss.item()

    print("epoch:",epoch,"loss:",total_loss/len(loader))

100%|███████████████████████████████████████| 3718/3718 [07:59<00:00,  7.75it/s]


epoch: 0 loss: 0.891244976285839


100%|███████████████████████████████████████| 3718/3718 [08:02<00:00,  7.71it/s]


epoch: 1 loss: 0.7127572225748434


  1%|▌                                        | 46/3718 [00:06<08:23,  7.29it/s]

In [ ]:
############################################
# 11. 텍스트 생성
#
# 입력 prompt → 다음 단어 생성
############################################

In [ ]:
def generate(model,prompt,max_new_tokens=20):

    model.eval()

    tokens=encode(prompt)

    x=torch.tensor(tokens).unsqueeze(0).to(device)

    for _ in range(max_new_tokens):

        logits=model(x[:,-max_len:])

        next_token=torch.argmax(
            logits[:,-1,:],
            dim=-1
        )

        x=torch.cat([x,next_token.unsqueeze(0)],dim=1)

    result=[itos[i] for i in x[0].tolist()]

    return " ".join(result)


print(generate(model,"i am"))